# Budget Forcing

**Paper**: [s1: Simple test-time scaling](https://arxiv.org/abs/2501.19393)

**Authors**: Niklas Muennighoff, Zitong Yang, Weijia Shi, Xiang Lisa Li, Li Fei-Fei, Hannaneh Hajishirzi, Luke Zettlemoyer, Percy Liang, Emmanuel Candes, Tatsunori Hashimoto

Budget forcing controls the length of a reasoning model's thinking at test time. It caps the thinking phase at a token budget and can either shorten reasoning (force the closing think tag once the budget is hit) or lengthen it (append an extension such as "Wait" to prompt continued reasoning) before generating the final answer.

Budget forcing is a decoding driver built on the generic phased driver: a bounded thinking phase, optional extension rounds, a forced closing tag, and an unbounded answer phase.

The method assumes a reasoning model. The thinking-phase boundary is the model's own closing think tag, so the driver can only find that boundary if the model actually emits one; on a non-reasoning model the tag never appears and the method degenerates to blind truncation plus a pasted-in tag.

## Method parameters

| parameter | type | description |
| --------- | ---- | ----------- |
| `max_thinking_tokens` | `int` | Token budget for each thinking segment |
| `extension_text` | `str` | Text appended to prolong reasoning |
| `num_extensions` | `int` | Number of extension rounds (0 disables) |
| `end_think` | `str` | The closing-think marker |

## Setup

If running this from a Google Colab notebook, uncomment the clone cell below. It is not necessary when running from a virtual environment where the package is already installed.

In [1]:
# !git clone https://github.com/IBM/AISteer360.git
# %cd AISteer360

The following authentication steps may be necessary to access any gated models (after being granted access by Hugging Face). Uncomment the following if you need to log in to the Hugging Face Hub:

In [2]:
# !pip install -q python-dotenv
# from dotenv import load_dotenv
# import os

# load_dotenv()
# token = os.getenv("HUGGINGFACE_TOKEN")
# from huggingface_hub import login
# login(token=token)

## Example: dialing a reasoning model's thinking budget

We use `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`, a small open reasoning model. Its chat template opens the thinking block (the prompt ends with `<think>`), and the model closes it by emitting `</think>` before writing its final answer, so the driver's boundary marker occurs naturally in every generation. Following the model card we sample with temperature 0.6 and top-p 0.95 rather than decoding greedily, with a fixed seed so runs are comparable.

In [3]:
import re

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from aisteer360.algorithms.core.steering_pipeline import SteeringPipeline
from aisteer360.algorithms.output_control.budget_forcing.control import BudgetForcing

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
END_THINK = "</think>"
SAMPLING = {"do_sample": True, "temperature": 0.6, "top_p": 0.95}

/dccstor/principled_ai/users/erikmiehling/AISteer360/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We define two helps (to split a generation into its thinking span and final answer, and the other to count thinking tokens). The split is on the first closing tag, so whatever the model generates after the (possibly forced) tag counts as answer, and when a generation runs out of tokens before any tag appears, the whole stream counts as thinking.

In [4]:
def split_thinking(text: str, end_think: str = END_THINK) -> tuple[str, str]:
    if end_think in text:
        thinking, answer = text.split(end_think, 1)
        return thinking, answer.strip()
    return text, ""


def num_tokens(tokenizer, text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

### Baseline: the model's natural thinking length

First, how the model behaves unforced. We generate with a plain `model.generate` call and a generous token limit, then measure how long the model chooses to think on a short multi-step word problem (the correct answer is 5).

In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

prompt = (
    "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. "
    "Her parents give her $15 for that purpose, and her grandparents give twice as much as her parents. "
    "How much more money does Betty need, in dollars?"
)
chat = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(chat, return_tensors="pt", add_special_tokens=False).to(model.device)

set_seed(42)
baseline_ids = model.generate(**inputs, max_new_tokens=2048, pad_token_id=tokenizer.eos_token_id, **SAMPLING)
baseline_text = tokenizer.decode(baseline_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

thinking, answer = split_thinking(baseline_text)
print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")
print(f"\nanswer: {answer}")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/339 [00:01<08:47,  1.56s/it]


Loading weights:   2%|▏         | 6/339 [00:01<01:10,  4.75it/s]


Loading weights:   5%|▌         | 18/339 [00:01<00:19, 16.59it/s]


Loading weights:   9%|▉         | 30/339 [00:01<00:10, 29.06it/s]

...


Loading weights:  92%|█████████▏| 311/339 [00:11<00:01, 26.64it/s]


Loading weights:  93%|█████████▎| 316/339 [00:11<00:01, 21.97it/s]


Loading weights:  97%|█████████▋| 328/339 [00:11<00:00, 28.97it/s]


Loading weights: 100%|██████████| 339/339 [00:11<00:00, 28.36it/s]

thinking tokens: 895

answer: Betty wants to buy a wallet that costs $100. She currently has half of that amount, which is $50. Her parents give her $15, and her grandparents give her twice as much as her parents, which is $30. 

First, we add the money given by her parents:
\[ 50 + 15 = 65 \]

Next, we add the money given by her grandparents:
\[ 65 + 30 = 95 \]

Finally, we subtract the total amount Betty has from the cost of the wallet to find out how much more she needs:
\[ 100 - 95 = 5 \]

Thus, Betty needs \boxed{5} more dollars.


Left alone, the model spends a substantial thinking budget on this problem before committing to an answer. That natural length is the reference point for everything below: shortening means cutting below it, extending means pushing past where the model would have stopped.

### Shortening: cap the budget and force the tag

We cap thinking at `max_thinking_tokens=64` with no extensions. The plan is a thinking phase that stops at the closing tag or at 64 tokens (whichever comes first), the forced `</think>`, then the answer phase. At 64 tokens the model is still mid-thought, so the thinking span below ends abruptly where the tag was pasted in. This is the "shorten" half of s1.

In [6]:
budget_forcing = BudgetForcing(max_thinking_tokens=64, num_extensions=0, end_think=END_THINK)

pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[budget_forcing],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline.steer()

set_seed(42)
output = pipeline.generate(
    input_ids=inputs["input_ids"].to(pipeline.model.device),
    max_new_tokens=512,
    pad_token_id=tokenizer.eos_token_id,
    **SAMPLING,
)
forced_text = tokenizer.decode(output[0], skip_special_tokens=True)
thinking, answer = split_thinking(forced_text)

print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")
print(f"\nend of thinking span: ...{thinking[-160:]}")
print(f"\nanswer: {answer}")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/339 [00:00<01:07,  5.03it/s]


Loading weights:   1%|          | 2/339 [00:00<01:09,  4.82it/s]


Loading weights:   8%|▊         | 27/339 [00:00<00:04, 70.92it/s]


Loading weights:  19%|█▉        | 64/339 [00:00<00:01, 153.65it/s]

...


Loading weights:  81%|████████  | 273/339 [00:01<00:00, 271.41it/s]


Loading weights:  89%|████████▉ | 302/339 [00:01<00:00, 275.33it/s]


Loading weights:  98%|█████████▊| 331/339 [00:01<00:00, 204.07it/s]


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 197.73it/s]

thinking tokens: 64

end of thinking span: ...et me figure out how much she currently has and how much more she needs. 

First, the problem says Betty has only half of the money she needs. So, if the wallet

answer: To determine how much more money Betty needs, let's break down her current savings.

1. The total cost of the wallet is $100.
2. Betty has half of that, which is $50.
3. Her parents give her $15, so adding that to her current savings: $50 + $15 = $65.
4. Her grandparents give twice as much as her parents, so that's 2 * $15 = $30.
5. Adding the grandparents' contribution: $65 + $30 = $95.
6. Subtracting the total she has ($95) from the wallet's cost ($100) gives $5 needed.

So, Betty needs $5 more.

**Answer:** Betty needs $\boxed{5}$ dollars more.


The thinking span stops mid-sentence at exactly the budget, and the model is forced to answer from whatever partial reasoning it has. Notice how the model compensates: the "answer" it writes after the forced tag quietly re-derives the whole solution instead of trusting the truncated thought. Cutting the thinking budget moved the reasoning; it did not remove it.

### Extending: append "Wait" and keep thinking

Extensions are the "lengthen" half of s1. Each extension round appends `Wait` to the stream and opens another bounded thinking segment, so a thought the budget would have cut short gets prolonged instead. We keep the per-segment budget at 128 tokens so the splice points are easy to locate: the driver appends `Wait` right after tokens 128 and 257 of the continuation.

In [7]:
budget_forcing = BudgetForcing(
    max_thinking_tokens=128,
    extension_text="Wait",
    num_extensions=2,
    end_think=END_THINK,
)

pipeline = SteeringPipeline(
    model_name_or_path=MODEL_NAME,
    controls=[budget_forcing],
    device_map="auto",
    hf_model_kwargs={"dtype": "auto"},
)
pipeline.steer()

set_seed(42)
output = pipeline.generate(
    input_ids=inputs["input_ids"].to(pipeline.model.device),
    max_new_tokens=512,
    pad_token_id=tokenizer.eos_token_id,
    **SAMPLING,
)
extended_text = tokenizer.decode(output[0], skip_special_tokens=True)
thinking, answer = split_thinking(extended_text)
print(f"thinking tokens: {num_tokens(tokenizer, thinking)}")

wait_len = num_tokens(tokenizer, "Wait")
out_ids = output[0]
for i, splice_at in enumerate([128, 128 + wait_len + 128], start=1):
    window = tokenizer.decode(out_ids[splice_at - 20:splice_at + wait_len + 20], skip_special_tokens=True)
    print(f"\nsplice {i}: ...{window}...")

print(f"\nanswer: {answer}")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/339 [00:00<01:06,  5.05it/s]


Loading weights:   1%|          | 2/339 [00:00<01:09,  4.85it/s]


Loading weights:   9%|▊         | 29/339 [00:00<00:03, 79.61it/s]


Loading weights:  18%|█▊        | 60/339 [00:00<00:01, 145.33it/s]

...


Loading weights:  81%|████████▏ | 276/339 [00:01<00:00, 278.42it/s]


Loading weights:  90%|████████▉ | 305/339 [00:01<00:00, 273.87it/s]


Loading weights:  99%|█████████▉| 336/339 [00:01<00:00, 283.09it/s]


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 212.74it/s]

thinking tokens: 385

splice 1: ...0 is 50. So, Betty currently has $50. 

Alright, that'sWait, no, hold on. It says her parents give her $15, and her grandparents give...

splice 2: .... Together, that's $15 plus $30. Let me add those up. Wait, 15 plus 30 is 45. So, altogether, Betty gets $...

answer: Betty needs a total of $100 for the wallet. She currently has half of this amount, which is $50. Her parents give her $15, and her grandparents give twice as much as her parents, so that's $30. Together, her parents and grandparents give her a total of $15 + $30 = $45. 

Adding her $50 to the $45 she received from her parents and grandparents, Betty has a total of $95. Therefore, she still needs $100 - $95 = $5 more. 

\boxed{5}


Each splice shows the same pattern: the segment is cut mid-thought at its budget, the appended `Wait` lands, and the model picks the reasoning back up, often by re-examining what it had just concluded. The total thinking length is now set by the driver, not by when the model felt done.

### The s1 story: answer quality vs. thinking budget

Budget forcing is the mechanism behind s1's test-time scaling curves, where answer quality is a function of allotted thinking compute. The sweep below runs the same problem at three budgets and tabulates the thinking tokens actually used and the final answer.

In [8]:
results = []
for budget in [64, 256, 1024]:
    sweep_pipeline = SteeringPipeline(
        model_name_or_path=MODEL_NAME,
        controls=[BudgetForcing(max_thinking_tokens=budget, num_extensions=0, end_think=END_THINK)],
        device_map="auto",
        hf_model_kwargs={"dtype": "auto"},
    )
    sweep_pipeline.steer()
    set_seed(42)
    output = sweep_pipeline.generate(
        input_ids=inputs["input_ids"].to(sweep_pipeline.model.device),
        max_new_tokens=max(512, budget),
        pad_token_id=tokenizer.eos_token_id,
        **SAMPLING,
    )
    thinking, answer = split_thinking(tokenizer.decode(output[0], skip_special_tokens=True))
    numbers = re.findall(r"-?\d+", answer)
    results.append((budget, num_tokens(tokenizer, thinking), numbers[-1] if numbers else answer[:40]))

print(f"{'budget':>7}  {'thinking tokens':>16}  {'final answer':>13}")
for budget, used, final_answer in results:
    print(f"{budget:>7}  {used:>16}  {final_answer:>13}")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


Loading weights:   1%|          | 2/339 [00:00<00:28, 11.66it/s]


Loading weights:   9%|▊         | 29/339 [00:00<00:02, 117.55it/s]


Loading weights:  19%|█▉        | 64/339 [00:00<00:01, 202.44it/s]


Loading weights:  27%|██▋       | 90/339 [00:00<00:01, 215.14it/s]

...


Loading weights:  79%|███████▉  | 267/339 [00:01<00:00, 264.23it/s]


Loading weights:  88%|████████▊ | 300/339 [00:01<00:00, 281.35it/s]


Loading weights:  97%|█████████▋| 329/339 [00:01<00:00, 275.23it/s]


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 211.42it/s]

 budget   thinking tokens   final answer
     64                64              5
    256               256              5
   1024               895              5


The thinking-token column tracks the budget exactly, which is the compute half of the s1 curve. On this problem the answer half is flat: every budget lands on the correct value, because when thinking is cut hard the model finishes the derivation inside its answer phase instead (visible in the shortened run above). Extra budget here buys directness rather than correctness. On problems at the edge of the model's ability, the same dial moves accuracy, which is the s1 result.

### Mechanics

`BudgetForcing` is a preset of the generic phased driver. Its per-example plan is:

- `Generated(until="</think>", budget=max_thinking_tokens)`, the bounded thinking phase
- `num_extensions` repetitions of `Fixed(extension_text)` followed by another bounded `Generated`
- `Fixed("</think>")`, the forced closing tag
- `Generated()`, the unbounded answer phase

Phase boundaries are substring stops on the closing marker only; the opening `<think>` plays no role in the mechanics (here it lives in the prompt, courtesy of the chat template). `Fixed` phases are plain token appends, so when the model closes its thinking naturally within budget, the forced tag still lands and the stream carries the tag twice. Plans are built per example, and batched inputs are handled by looping over rows. Every `Generated` phase delegates to `model.generate` with the pipeline's composed stacks, so a step-level control (for example RAD) steers each phase, including the extensions.

### Takeaway

Budget forcing turns thinking length into an inference-time dial: one integer trades answer quality against decode compute, and the "Wait" trick buys extra reasoning on demand without touching weights or prompts. It only makes sense on models that already externalize their reasoning between think tags.

[phased_decoding.ipynb](../generics/phased_decoding.ipynb) demonstrates the generic this preset is built on, including a thinking-intervention plan that splices steering text into the reasoning stream rather than bounding its length. See the [output control](https://ibm.github.io/AISteer360/concepts/controls/#output-control) section of the docs for the full family.